# CS781 capstone - Phase 2: Legal named entities (L-NER)
Donkena Manideep (261040623).  Runs the code of `phase2/` from my submission on Kaggle.

**Before running:** turn on a GPU (T4 x2 or better; the code uses bfloat16), turn on Internet (for the weights and
`pip`), and attach as Kaggle datasets: (a) the code (the `code/` folder of my archive) unless the GitHub repository is
public, (b) the Phase 2 competition data.
Set `DATA_INPUT` below to the folder of (b).  `QUICK = True` runs short smoke tests; `False` runs the full recipe
(the full runs took 15-95 minutes each on an RTX PRO 5000 and take several times longer on a T4).

In [ ]:
PHASE = 2
QUICK = True                                    # False = the full recipe
DATA_INPUT = "/kaggle/input/cs781-phase2-data/Phase_2_LNER"   # <- the attached competition data
DATA_SUBDIR = "Phase_2_LNER"
REPO_URL = "https://github.com/ManideepDonkena/CS781-Legal-Intelligence-System"
REL = "https://github.com/ManideepDonkena/CS781-Legal-Intelligence-System/releases/download/models-v1/"

In [ ]:
# ---- setup: code, weights, folder layout ----------------------------------------------------------------------
import glob, os, shutil, subprocess, sys, urllib.request

def sh(cmd):
    """Run a shell command, print its output as it comes, stop on error."""
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")

# 1. the code: an attached Kaggle dataset that holds phase1..phase7 (the unzipped archive's code/ folder),
#    else a clone of the GitHub repository (the repository must be public for this)
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/phase1/final_model", recursive=True)]
if found:
    REPO = os.path.dirname(found[0])
else:
    REPO = "/kaggle/working/repo"
    if not os.path.isdir(REPO):
        sh(f"git clone --depth 1 {REPO_URL} {REPO}")
print("code:", REPO)
sh(f"pip install -q -r {REPO}/requirements.txt || true")

# 2. the layout the code expects
CAP = "/home/mani/cs781/Capstone"
LLM = "/home/mani/cs781_phase1/Legal_LLM"
os.makedirs(os.path.dirname(LLM), exist_ok=True)
if not os.path.exists(LLM):
    shutil.copytree(f"{REPO}/phase1/final_model", LLM)                 # model code (modern_arch.py, rope.py, ...)
NAMES = {2: "Phase2_LNER", 3: "Phase3_RR", 4: "Phase4_CJPE", 5: "Phase5_LSI", 6: "Phase6_SUM", 7: "Phase7_PCR"}
for p, name in NAMES.items():                                          # every phase's src (they import each other)
    os.makedirs(f"{CAP}/{name}/runs", exist_ok=True)
    if not os.path.exists(f"{CAP}/{name}/src"):
        shutil.copytree(f"{REPO}/phase{p}/src", f"{CAP}/{name}/src")
if not os.path.exists(f"{CAP}/shared"):
    shutil.copytree(f"{REPO}/shared", f"{CAP}/shared")

# 3. this phase's competition data (attach it as a Kaggle dataset; DATA_INPUT is its folder)
if PHASE > 1:
    target = f"{CAP}/{NAMES[PHASE]}/data/{DATA_SUBDIR}"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        os.symlink(DATA_INPUT, target)
    print("data:", target, "->", os.listdir(target)[:10])

# 4. the Phase 1 weights from the GitHub release models-v1 (about 1.2 GB)
W = "/workspace/models/cs781_phase1_runs/ROPEFIX_295M"
os.makedirs(W, exist_ok=True)
for name, dest in [("phase1_ROPEFIX_295M_weights.pt", "best.pt"), ("phase1_tokenizer.json", "tokenizer.json"),
                   ("phase1_run_config.json", "run_config.json")]:
    if not os.path.exists(f"{W}/{dest}"):
        print("downloading", name, flush=True)
        urllib.request.urlretrieve(REL + name, f"{W}/{dest}")
sys.path[:0] = [LLM, f"{CAP}/shared"]
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| bf16:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## Main recipe (README of phase 2)

In [ ]:
q = "--epochs 1 --max_steps 20" if QUICK else ""
sh(f"cd {CAP}/Phase2_LNER/src && python train_ner.py --backbone {W} --kfold 5:0 --out_dir ../runs/nb_k0 {q}")

## Fine-tuning methods, LoRMA and distillation (9 Nov study, 5 held-out splits)
`adapters.py` uses my Assignment 4 LoRMA functions (`q4_peft_lorma.py`, unchanged).

In [ ]:
q = "--epochs 1 --max_steps 20" if QUICK else ""
for method in ["frozen", "full", "lora", "lorma"]:
    sh(f"cd {CAP}/Phase2_LNER/src && python train_adapt.py --backbone {W} --method {method} --kfold 5:0 --out_dir ../runs/adapt/{method}_k0 {q}")
sh(f"cd {CAP}/Phase2_LNER/src && python summarize_adapt.py")

## Scores

In [ ]:
import json
for f in sorted(glob.glob(f"{CAP}/Phase2_LNER/runs/**/log.json", recursive=True)):
    j = json.load(open(f)); print(f.split("/runs/")[1], j.get("macro_f1"))